# 44 — Blood: stratification and heterogeneity of the malignant-vs-reactive CD4 axis

43 asks whether an antigen separates tumour from reactive CD4 **on average**. That average is
made of 52 patients with different diseases, different treatments, different ages and very
different tumour burdens, and CTCL is heterogeneous enough that an average can describe nobody.
This notebook asks the two questions that follow from it:

1. **Does the effect depend on who the patient is?** — MF vs Sézary, treatment at sampling,
   sex, age, tumour burden.
2. **Is the effect the same size in everyone, or large in a subset?** — random-effects
   heterogeneity, the subgroup statistics built for exactly this (COPA, the outlier sum),
   bimodality, donor clustering, and the eligibility curve a biomarker-selected trial is
   designed against.

**Power first.** The cohort is 52 independent paired donors: 37 SS, 8 MF, 5 unresolved, 2
MF/SS. Treatment is almost entirely confounded with study, there are 2 early-stage donors
against 35 advanced, and response and survival are essentially absent. §1 prints the counts
before anything is tested, and every contrast is placed in one of three tiers:

| tier | meaning |
|---|---|
| **1** | enough donors to test, and the test is run |
| **2** | enough to describe, with effect sizes and no claim |
| **3** | not testable — reported as a count, so the gap is visible rather than silently absent |

Nothing here re-derives the screen. It reads the cleaned cohort and the rebuilt tables that 43
produces, so **run 43 first** (or at least `jobs/run_target_screen_qc.sh --tissue blood`).

In [ ]:
# ============================================================================
# §0  Setup
# ============================================================================
import warnings; warnings.filterwarnings("ignore")
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm


def _resolve_nb_dir() -> Path:
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(f"could not locate MF/data from {start}")


NB_DIR = _resolve_nb_dir()
sys.path.insert(0, str(NB_DIR / "helpers"))

import surface_targets as ST
import target_screen as TS
import target_cohort as TC
import target_qc as TQ
import target_figs as TF
import target_het as TH
import cluster_malig_helpers as CM
import malignancy as MAL

CM.use_tissue("blood")
MAL.use_tissue("blood")
TQ.use_tissue("blood")           # 42/47/48 set it to skin, and the module keeps the last value

TISSUE = "blood"
TARGETS = NB_DIR / "data" / "targets"
PREFIX = "tgt44_"
MALIG, REACT, HC_CD4 = "CD4_malignant", "CD4_reactive", "CD4_HC"
MIN_CELLS, MIN_DONORS, DETECT_FLOOR = 20, 5, 0.10
N_PERM = 1000      # sign-flip permutations for the COPA / outlier-sum null

ST.TAB_PREFIX = ST.FIG_PREFIX = PREFIX
TF.rc()
pd.set_option("display.width", 240)

don = TC.blood_donor_table()
long = TC.filter_long(pd.read_parquet(TARGETS / f"{TISSUE}_target_long.parquet"), don)
meta = TC.filter_long(pd.read_parquet(TARGETS / f"{TISSUE}_target_meta.parquet"), don)
man = json.loads((TARGETS / f"{TISSUE}_target_manifest.json").read_text())
assert man.get("builder") == "run_target_screen_qc.py", (
    "run jobs/run_target_screen_qc.sh --tissue blood first -- this notebook needs sd_all for "
    "the random-effects model and mean_all on the true library")
bur = TC.burden(long)
don = don.join(bur, how="left")

paired = meta[meta["arm"].isin([MALIG, REACT])].groupby("donor")["arm"].nunique().eq(2)
PAIRED = sorted(paired.index[paired])
DON = don.loc[PAIRED]
DONOR_STUDY = dict(zip(meta["donor"], meta["study"]))

sel = TS.paired_selectivity(long, MALIG, REACT, value="mean_all", min_donors=MIN_DONORS)
prev = TS.prevalence(long, MALIG, detect_floor=DETECT_FLOOR)

# The gene set everything below runs on. 43's shortlist is the headline, but restricting the
# heterogeneity analysis to it would answer only "are my winners consistent" -- a gene that is
# flat on average because it is strongly up in a third of patients is invisible in that list
# and is exactly what this notebook exists to find. So: every gene that reaches the paired
# test, with the shortlist marked.
SHORT = pd.read_csv(ST.TAB_DIR / "tgt43_shortlist_blood.csv")["gene"].tolist() \
    if (ST.TAB_DIR / "tgt43_shortlist_blood.csv").exists() else []
GENES = sel["gene"].tolist()
print(f"NB_DIR = {NB_DIR}")
print(f"{len(PAIRED)} paired donors | {len(GENES):,} genes tested | "
      f"{len(SHORT)} on 43's shortlist")

## §1 — What this cohort can and cannot answer

Read the table before the sections. Every row is a candidate stratification; the counts decide
which tier it lands in, and the tier decides whether the result below is a test, a description
or an absence.

Two structural facts to keep in view:

- **Treatment is confounded with study.** ECP is 25 donors, but they are concentrated in buus
  and ren; bexarotene in ren is never given without ECP. The ECP contrast is therefore run
  study-blocked (van Elteren), and the within-study comparison is printed separately so the
  confounding is visible rather than averaged over.
- **Burden is the only covariate with complete coverage.** `clin_blood_tumor_pct` is curated for
  ~20 donors; the malignant share of the CD4 compartment is computable for all of them and is
  the better continuous axis. It is also the one 43 used as the contamination test, so a marker
  whose effect rides on burden should be read twice.

In [ ]:
# ============================================================================
# §1  The power table
# ============================================================================
FIELDS = {
    "dx2 (MF vs SS)": "dx2", "sex": "sex", "age": "age",
    "stage": "clin_stage", "stage class": "clin_stage_class", "B stage": "clin_B",
    "blood tumour %": "blood_tumor_pct", "treatment at sampling known": "tx_known",
    "treatment-naive": "clin_tx_naive", "prior lines": "n_prior_lines",
    "response": "clin_response", "outcome": "clin_outcome",
    "drug sensitivity": "clin_drug_sensitivity", "malignant burden": "burden",
}
rows = []
for label, col in FIELDS.items():
    if col not in DON.columns:
        rows.append({"field": label, "n": 0, "note": "column absent"}); continue
    v = DON[col]
    # A boolean column is never NA, so counting `notna` would report every donor as covered.
    have = v.astype(bool) if v.dtype == bool else v.notna()
    rec = {"field": label, "n": int(have.sum()),
           "MF": int(have[DON["dx2"].eq("MF")].sum()),
           "SS": int(have[DON["dx2"].eq("SS")].sum())}
    n = int(have.sum())
    if v.dtype == object and n:
        rec["levels"] = ", ".join(f"{k}:{c}" for k, c in v.value_counts().head(5).items())
    elif pd.api.types.is_numeric_dtype(v) and n:
        rec["levels"] = f"median {v.median():.3g}, range {v.min():.3g}-{v.max():.3g}"
    rows.append(rec)
power = pd.DataFrame(rows)
print(f"coverage over the {len(DON)} paired donors:")
print(power.to_string(index=False))

print("\ntreatment at sampling (a donor can be on several):")
tx = pd.DataFrame({k[3:]: DON[k] for k in DON.columns if k.startswith("tx_")
                   and k != "tx_known"}).sum().sort_values(ascending=False)
tx = tx[tx > 0].rename("donors").to_frame()
tx["MF"] = [int((DON[f"tx_{k}"] & DON["dx2"].eq("MF")).sum()) for k in tx.index]
tx["SS"] = [int((DON[f"tx_{k}"] & DON["dx2"].eq("SS")).sum()) for k in tx.index]
tx["studies"] = [", ".join(sorted(DON.loc[DON[f"tx_{k}"], "study"].unique())) for k in tx.index]
print(tx.to_string())

print("\nECP against study -- the confounding, as a table rather than a sentence:")
print(pd.crosstab(DON["study"], DON["tx_ecp"], dropna=False).to_string())

TIERS = {
    1: ["dx2 (MF vs SS)", "treatment: ECP", "sex", "age", "malignant burden"],
    2: ["treatment: HDACi", "treatment: interferon", "treatment: retinoid", "B stage",
        "prior lines"],
    3: ["stage (2 early vs 35 advanced)", "treatment-naive", "treatment: mogamulizumab",
        "bexarotene (never separable from ECP)", "response", "outcome", "survival"],
}
print("\n" + "=" * 78)
for t, items in TIERS.items():
    print(f"TIER {t}: " + "; ".join(items))
print("=" * 78)
power.to_csv(ST.tab("power_table"), index=False)
tx.to_csv(ST.tab("treatment_counts"))
print("wrote", ST.tab("power_table"), "and", ST.tab("treatment_counts"))

## §2 — Tier 1: the contrasts with enough donors to test

Each contrast is the same paired within-donor test as 43, run inside each stratum, plus a
two-sample comparison of the per-donor effects between strata (`p_interaction`). That
interaction is the question — not "is the antigen up in MF" but "is it *more* up in MF than in
SS" — and with 8 MF donors it is a consistency check, not a finding. The n for each side is
printed on every row so it cannot be read as more than that.

In [ ]:
# ============================================================================
# §2a  Disease: MF vs Sezary
# ============================================================================
dx_map = DON.loc[DON["dx2"].isin(["MF", "SS"]), "dx2"]
print(f"MF {int(dx_map.eq('MF').sum())} donors vs SS {int(dx_map.eq('SS').sum())} donors "
      f"({len(DON) - len(dx_map)} unresolved or MF/SS, excluded from this contrast)")

dxs = TS.strat_selectivity(long, MALIG, REACT, dx_map, value="mean_all", min_donors=3)
wide = dxs.pivot_table(index="gene", columns="stratum",
                       values=["log2fc_paired", "n_donors_paired", "frac_donors_up"])
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
inter = (dxs[["gene", "delta_lfc", "p_interaction", "q_interaction"]]
         .drop_duplicates("gene").set_index("gene"))
dx_tab = wide.join(inter).join(sel.set_index("gene")[["log2fc_paired", "q"]]
                               .rename(columns={"log2fc_paired": "log2fc_all",
                                                "q": "q_all"}))
dx_tab = dx_tab.reset_index()
dx_tab["on_shortlist"] = dx_tab["gene"].isin(SHORT)

sig = dx_tab[dx_tab["q_interaction"] < 0.05].sort_values("delta_lfc")
print(f"\ngenes whose effect differs between MF and SS at q < 0.05: {len(sig)}")
_c = ["gene", "log2fc_paired_MF", "n_donors_paired_MF", "log2fc_paired_SS",
      "n_donors_paired_SS", "delta_lfc", "q_interaction", "log2fc_all", "on_shortlist"]
print(sig[_c].to_string(index=False, float_format=lambda v: f"{v:.3g}")
      if len(sig) else "  none -- with 8 MF donors this is the expected result, and it is not "
      "evidence that the two diseases behave the same")

print("\nthe shortlist, split by disease (no claim; read the two n columns):")
print(dx_tab[dx_tab["on_shortlist"]].sort_values("log2fc_all", ascending=False)[_c]
      .to_string(index=False, float_format=lambda v: f"{v:.3g}"))

# Within-study, where the two diseases actually sit side by side. Everywhere else, disease and
# study are the same variable.
print("\n" + "=" * 78)
print("within-study MF vs SS, the only places the contrast is not confounded with cohort:")
for study in sorted(DON["study"].unique()):
    sub = DON[DON["study"].eq(study) & DON["dx2"].isin(["MF", "SS"])]
    if sub["dx2"].nunique() < 2 or sub["dx2"].value_counts().min() < 2:
        continue
    print(f"\n  {study}: {sub['dx2'].value_counts().to_dict()}")
    s = TS.strat_selectivity(
        long[long["donor"].isin(sub.index) & long["gene"].isin(SHORT[:12])],
        MALIG, REACT, sub["dx2"], value="mean_all", min_donors=2, interaction=False)
    print(s.pivot_table(index="gene", columns="stratum", values="log2fc_paired")
          .round(2).to_string())

dx_tab.to_csv(ST.tab("strat_disease"), index=False)
print("\nwrote", ST.tab("strat_disease"))

In [ ]:
# ============================================================================
# §2b  Treatment at sampling: ECP, the only agent with two usable arms
# ============================================================================
# Extracorporeal photopheresis. 25 of the paired donors are on it, 20 are known not to be, and
# the split is almost a study split -- so the headline test is the study-blocked one
# (van Elteren), not the pooled one. Both are printed.
ecp_known = DON[DON["tx_known"]]
ecp_map = ecp_known["tx_ecp"].map({True: "ECP", False: "no ECP"})
print(f"{int((ecp_map == 'ECP').sum())} on ECP vs {int((ecp_map == 'no ECP').sum())} not, "
      f"{len(DON) - len(ecp_map)} with treatment unknown")

ecp = TS.strat_selectivity(long, MALIG, REACT, ecp_map, value="mean_all", min_donors=3)
ew = ecp.pivot_table(index="gene", columns="stratum",
                     values=["log2fc_paired", "n_donors_paired"])
ew.columns = [f"{a}_{b}".replace(" ", "_") for a, b in ew.columns]
ew = ew.join(ecp[["gene", "delta_lfc", "p_interaction", "q_interaction"]]
             .drop_duplicates("gene").set_index("gene")).reset_index()
ew["on_shortlist"] = ew["gene"].isin(SHORT)

sig = ew[ew["q_interaction"] < 0.05].sort_values("delta_lfc")
print(f"\ngenes whose malignant-vs-reactive effect differs by ECP status at q < 0.05: "
      f"{len(sig)}")
_c = ["gene", "log2fc_paired_ECP", "n_donors_paired_ECP", "log2fc_paired_no_ECP",
      "n_donors_paired_no_ECP", "delta_lfc", "q_interaction", "on_shortlist"]
print(sig[_c].head(40).to_string(index=False, float_format=lambda v: f"{v:.3g}")
      if len(sig) else "  none")

print("\nthe shortlist by ECP status:")
print(ew[ew["on_shortlist"]][_c].to_string(index=False, float_format=lambda v: f"{v:.3g}"))

print("\n" + "=" * 78)
print("study-blocked version of the same contrast (van Elteren), which is the one to quote:")
blk = TS.blocked_selectivity(
    long[long["donor"].isin(ecp_known.index)].assign(
        arm=lambda d: d["arm"].where(~d["arm"].eq(MALIG),
                                     d["donor"].map(ecp_map).radd("MALIG_"))),
    "MALIG_ECP", "MALIG_no ECP", block=DONOR_STUDY, value="mean_all", min_per_arm=3)
print(f"{len(blk):,} genes testable with >=3 donors per arm per study block, "
      f"{int((blk['q_blocked'] < 0.05).sum())} at q < 0.05")
print(blk.sort_values("z_blocked").head(15).to_string(index=False,
                                                      float_format=lambda v: f"{v:.3g}"))
print("...")
print(blk.sort_values("z_blocked", ascending=False).head(15)
      .to_string(index=False, float_format=lambda v: f"{v:.3g}"))

print("\n*** This compares the malignant arm of ECP-treated donors against the malignant arm "
      "of untreated donors, which is a between-patient comparison and loses the within-donor "
      "pairing. Bexarotene cannot be separated from ECP anywhere in this cohort (every ren "
      "donor on bexarotene is also on ECP), so 'ECP' here means 'ECP, usually with a "
      "retinoid'. ***")
ew.to_csv(ST.tab("strat_ecp"), index=False)
blk.to_csv(ST.tab("strat_ecp_blocked"), index=False)
print("\nwrote", ST.tab("strat_ecp"), "and", ST.tab("strat_ecp_blocked"))

In [ ]:
# ============================================================================
# §2c  Everything at once: per-gene regression of the donor effect on who the donor is
# ============================================================================
# One model per gene, over donors: does the within-donor effect depend on diagnosis, ECP,
# sex, age or tumour burden. Study is left out of the design -- with 10 studies and 52 donors
# it would absorb the treatment term it is confounded with -- and is reported separately as
# the variance it explains on its own.
M = TH.lfc_matrix(long, MALIG, REACT, min_donors=MIN_DONORS)
X = pd.DataFrame(index=M.columns)
X["is_MF"] = DON["dx2"].reindex(X.index).eq("MF").astype(float)
X["ecp"] = DON["tx_ecp"].reindex(X.index).where(
    DON["tx_known"].reindex(X.index)).astype(float)
X["is_F"] = DON["sex"].reindex(X.index).eq("F").astype(float)
X["age_z"] = ((DON["age"].reindex(X.index) - DON["age"].mean()) / DON["age"].std())
X["burden_z"] = ((DON["burden"].reindex(X.index) - DON["burden"].mean())
                 / DON["burden"].std())
X = X.dropna()
print(f"{len(X)} donors with every covariate non-missing "
      f"(of {M.shape[1]}); terms: {list(X.columns)}")
print(X.describe().round(3).to_string())

Xd = sm.add_constant(X)
rows = []
for gene, y in M.loc[:, X.index].iterrows():
    y = y.dropna()
    if len(y) < 20:
        continue
    fit = sm.OLS(y.to_numpy(float), Xd.loc[y.index]).fit()
    rec = {"gene": gene, "n": len(y), "r2": fit.rsquared}
    for t in X.columns:
        rec[f"b_{t}"] = fit.params[t]
        rec[f"p_{t}"] = fit.pvalues[t]
    rows.append(rec)
reg = pd.DataFrame(rows)
for t in X.columns:
    reg[f"q_{t}"] = multipletests(reg[f"p_{t}"], method="fdr_bh")[1]
reg["on_shortlist"] = reg["gene"].isin(SHORT)

print(f"\n{len(reg):,} genes modelled. Terms surviving BH at q < 0.05:")
print(pd.Series({t: int((reg[f"q_{t}"] < 0.05).sum()) for t in X.columns}).to_string())

for t in X.columns:
    s = reg[reg[f"q_{t}"] < 0.05].reindex(reg[f"b_{t}"].abs().sort_values(ascending=False).index)
    s = s.dropna(subset=["gene"]).head(15)
    print(f"\n--- {t}: {len(reg[reg[f'q_{t}'] < 0.05])} genes, strongest 15")
    if len(s):
        print(s[["gene", f"b_{t}", f"q_{t}", "r2", "n", "on_shortlist"]]
              .to_string(index=False, float_format=lambda v: f"{v:.3g}"))
    else:
        print("  none")

print("\nstudy on its own, as a one-way ANOVA per gene (not in the model above):")
st = DON["study"].reindex(M.columns)
rows = []
for gene, y in M.iterrows():
    y = y.dropna()
    g = st.reindex(y.index)
    groups = [y[g.eq(s)].to_numpy() for s in g.dropna().unique() if (g.eq(s)).sum() >= 3]
    if len(groups) < 3:
        continue
    rows.append({"gene": gene, "p_study": float(stats.f_oneway(*groups).pvalue)})
anv = pd.DataFrame(rows)
anv["q_study"] = multipletests(anv["p_study"], method="fdr_bh")[1]
print(f"  {int((anv['q_study'] < 0.05).sum()):,} of {len(anv):,} genes differ by study at "
      "q < 0.05 -- the scale of the batch term the model above does not carry.")

reg.merge(anv, on="gene", how="left").to_csv(ST.tab("strat_regression"), index=False)
print("\nwrote", ST.tab("strat_regression"))

## §3 — Tiers 2 and 3

Tier 2 is described, not tested: six donors on an HDAC inhibitor can produce an effect size and
a confidence interval, and nothing that should survive a multiple-testing correction. Tier 3 is
printed as counts only. A stratification that is absent from a notebook reads as a
stratification that was not interesting; one that is printed with `n = 1` reads as what it is.

In [ ]:
# ============================================================================
# §3  Tier 2 described, tier 3 counted
# ============================================================================
def describe_split(mask, label, genes, min_side=3):
    """Median per-donor effect on each side of a binary split, with a CI and no test."""
    m = mask.dropna().astype(bool)
    a, b = m.index[m], m.index[~m]
    if min(len(a), len(b)) < min_side:
        return pd.DataFrame()
    rows = []
    for gene in genes:
        if gene not in M.index:
            continue
        ya, yb = M.loc[gene, M.columns.intersection(a)], M.loc[gene, M.columns.intersection(b)]
        ya, yb = ya.dropna(), yb.dropna()
        if min(len(ya), len(yb)) < min_side:
            continue
        boot = [np.median(np.random.default_rng(i).choice(ya, len(ya)))
                - np.median(np.random.default_rng(i + 7).choice(yb, len(yb)))
                for i in range(400)]
        rows.append({"split": label, "gene": gene, "n_yes": len(ya), "n_no": len(yb),
                     "median_yes": float(np.median(ya)), "median_no": float(np.median(yb)),
                     "delta": float(np.median(ya) - np.median(yb)),
                     "ci_lo": float(np.percentile(boot, 2.5)),
                     "ci_hi": float(np.percentile(boot, 97.5))})
    return pd.DataFrame(rows)


TIER2 = {
    "HDACi at sampling": DON["tx_hdaci"].where(DON["tx_known"]),
    "interferon at sampling": DON["tx_interferon"].where(DON["tx_known"]),
    "retinoid at sampling": DON["tx_retinoid"].where(DON["tx_known"]),
    "ever on ECP": DON["ever_ecp"],
    "B2 (vs B0/B1)": DON["clin_B"].map(lambda v: v == "B2" if pd.notna(v) else np.nan),
    ">=3 prior lines": DON["n_prior_lines"].map(
        lambda v: v >= 3 if pd.notna(v) else np.nan),
    "cultured 2 days ex vivo": DON["is_cultured"],
    "shared TRB clone (buus PT05/06/18)": DON["clone_shared"],
}
t2 = pd.concat([describe_split(v, k, SHORT[:15]) for k, v in TIER2.items()],
               ignore_index=True)
print("TIER 2 -- effect sizes with bootstrap CIs, on 43's leading antigens. No test, no "
      "correction, and no row here is evidence of a treatment effect:\n")
for split, d in t2.groupby("split"):
    print(f"--- {split}  (n = {d['n_yes'].iloc[0]} vs {d['n_no'].iloc[0]})")
    print(d[["gene", "median_yes", "median_no", "delta", "ci_lo", "ci_hi"]]
          .sort_values("delta").to_string(index=False, float_format=lambda v: f"{v:.3g}"))
    print()
t2.to_csv(ST.tab("tier2_descriptive"), index=False)

print("=" * 78)
print("TIER 3 -- not testable in this cohort. The counts, so the gap is explicit:")
print("=" * 78)
t3 = []
t3.append(("stage, early vs advanced",
           DON["clin_stage_class"].value_counts().to_dict(),
           "2 early donors; the blood cohort is advanced disease by construction"))
t3.append(("treatment-naive", DON["clin_tx_naive"].value_counts().to_dict(),
           "harro's 'none' comes from an aggregator, so naive stays unknown there"))
for agent in ("mogamulizumab", "brentuximab", "alemtuzumab", "chemotherapy", "checkpoint",
              "alloSCT", "methotrexate"):
    n = int(DON[f"tx_{agent}"].sum())
    if n <= 4:
        t3.append((f"treatment: {agent}", {"on": n, "off": int(len(DON) - n)},
                   "single-donor case note at most"))
t3.append(("bexarotene separable from ECP",
           pd.crosstab(DON["tx_retinoid"], DON["tx_ecp"]).to_dict(),
           "every retinoid donor in ren is also on ECP"))
for col, name in (("clin_response", "response"), ("clin_outcome", "outcome"),
                  ("clin_drug_sensitivity", "drug sensitivity")):
    t3.append((name, DON[col].value_counts().to_dict(), "essentially absent"))
for name, counts, note in t3:
    print(f"  {name:38s} {counts}")
    print(f"  {'':38s} -> {note}")

print("\nmogamulizumab, as a single-donor case note on its own target:")
_mog = DON.index[DON["tx_mogamulizumab"]]
if len(_mog):
    _c = M.loc[M.index.intersection(["CCR4"]), M.columns.intersection(_mog)]
    print(_c.round(3).to_string())
    print(f"  cohort median CCR4 effect: {M.loc['CCR4'].median():.3f} "
          f"over {int(M.loc['CCR4'].notna().sum())} donors")
    print("  Loss of CCR4 is the documented resistance mechanism to mogamulizumab, so a low "
          "value in a treated donor is the expected direction. One donor cannot distinguish "
          "that from noise.")
else:
    print("  no paired donor is on mogamulizumab at sampling")
print("\nwrote", ST.tab("tier2_descriptive"))

## §4 — Heterogeneity: is the effect the same size in everyone?

Four views of the same donor × gene matrix, because they disagree and the disagreement is the
information.

1. **Random-effects meta-analysis.** Each donor contributes an effect and a standard error
   (delta method, from the per-arm cell SDs the QC build stores). `tau2` is the spread of the
   *true* effect once each donor's own sampling noise is removed, `I2` the share of observed
   spread that is not noise, and the prediction interval what the next patient would plausibly
   show. A large pooled effect with `I2` near 1 is a subgroup antigen with a high average.
2. **COPA and the outlier sum.** Built for genes activated in a fraction of tumours, which a
   mean test averages away. Both standardise by median and MAD and score the upper tail; the
   null is a sign flip of the paired differences, the same exchangeability the Wilcoxon assumes.
3. **Bimodality.** One- versus two-component Gaussian mixture BIC and the bimodality
   coefficient. (`diptest` is not installed in `neural_nmf_env`, so Hartigan's dip is not
   available.) With ~50 donors these are suggestive, not conclusive, and are reported beside
   `I2` rather than instead of it.
4. **Eligibility.** In what fraction of patients would at least X% of the tumour carry the
   antigen. This is the form a biomarker-selected trial is designed against, and the one that
   can be read without any of the above.

Then: do the donors fall into groups, and do those groups correspond to anything clinical.

In [ ]:
# ============================================================================
# §4a  Random-effects meta-analysis across donors
# ============================================================================
SE = TH.lfc_se(long, MALIG, REACT)
re_tab = TH.meta_random_effects(M, SE, min_donors=MIN_DONORS)
re_tab["on_shortlist"] = re_tab["gene"].isin(SHORT)
print("sampling variance source:", re_tab["se_source"].iloc[0])
print(f"{len(re_tab):,} genes pooled over up to {int(re_tab['n_donors'].max())} donors\n")

print(f"median I2 = {re_tab['I2'].median():.2f}. Each donor contributes thousands of cells, "
      "so the sampling term in the model is almost zero and nearly all the observed spread is "
      "counted as real: I2 therefore sits near 1 for almost every gene and does NOT separate "
      "them. `tau` -- the absolute between-patient SD of the effect, in log2FC units -- keeps "
      "its meaning, and the ranking below uses it.\n")

print("the most heterogeneous antigens with a positive pooled effect -- large on average, "
      "and not the same size in everyone:")
_c = ["gene", "n_donors", "mu", "se", "tau", "tau_over_mu", "I2", "pi_lo", "pi_hi",
      "frac_donors_up", "q_Q", "on_shortlist"]
het = re_tab[(re_tab["mu"] > 0.3)].sort_values("tau", ascending=False)
print(het[_c].head(25).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

print("\nthe most CONSISTENT antigens with a positive pooled effect -- the ones whose "
      "prediction interval stays above zero, which is what 'works in every patient' means:")
con = re_tab[(re_tab["mu"] > 0.3) & (re_tab["pi_lo"] > 0)].sort_values("mu", ascending=False)
print(con[_c].head(25).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print(f"\n{len(con)} of {int((re_tab['mu'] > 0.3).sum())} genes with a pooled effect above "
      "0.3 have a prediction interval that excludes zero.")

print("\n43's shortlist, ranked by how consistent it is (smallest tau first):")
print(re_tab[re_tab["on_shortlist"]].sort_values("tau")[_c]
      .to_string(index=False, float_format=lambda v: f"{v:.3g}"))
re_tab.to_csv(ST.tab("heterogeneity_meta"), index=False)
print("\nwrote", ST.tab("heterogeneity_meta"))

In [ ]:
# ============================================================================
# §4b  COPA and the outlier sum: up in a subset, by design
# ============================================================================
cp = TH.copa(M, pct=75, n_perm=N_PERM, min_donors=MIN_DONORS)
cp = cp.merge(re_tab[["gene", "mu", "I2"]], on="gene", how="left")
cp = cp.merge(sel[["gene", "log2fc_paired", "q"]], on="gene", how="left")
cp["on_shortlist"] = cp["gene"].isin(SHORT)

_c = ["gene", "copa_p75", "outlier_sum", "n_outlier_donors", "frac_outlier", "q_copa",
      "q_outlier_sum", "mu", "I2", "log2fc_paired", "q", "on_shortlist"]
_c = [c for c in _c if c in cp.columns]
print("highest COPA score -- extreme in a minority of patients:")
print(cp.sort_values("copa_p75", ascending=False).head(25)[_c]
      .to_string(index=False, float_format=lambda v: f"{v:.3g}"))

print("\nthe interesting cell of the table: a high outlier score and a cohort effect that is "
      "NOT significant. These are antigens a mean test discards and a biomarker-selected "
      "trial would not:")
hid = cp[(cp["q"] >= 0.05) & (cp.get("q_copa", pd.Series(1, index=cp.index)) < 0.1)]
hid = hid.sort_values("copa_p75", ascending=False).head(25)
print(hid[_c].to_string(index=False, float_format=lambda v: f"{v:.3g}")
      if len(hid) else "  none at these thresholds")

bm = TH.bimodality(M, min_donors=20)
bm = bm.merge(cp[["gene", "copa_p75", "frac_outlier"]], on="gene", how="left")
print("\nstrongest two-component structure in the per-donor effects "
      "(delta_bic > 0 favours two groups):")
print(bm.head(20).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print(f"\n{int(bm['two_component'].sum()):,} of {len(bm):,} genes prefer two components. With "
      f"{M.shape[1]} donors that is a weak instrument -- read it with I2 and the COPA column, "
      "not alone.")

cp.to_csv(ST.tab("heterogeneity_copa"), index=False)
bm.to_csv(ST.tab("heterogeneity_bimodality"), index=False)
print("\nwrote", ST.tab("heterogeneity_copa"), "and", ST.tab("heterogeneity_bimodality"))

In [ ]:
# ============================================================================
# §4c  Do the donors fall into groups, and is any of it clinical
# ============================================================================
HG = [g for g in (SHORT + het["gene"].head(40).tolist()) if g in M.index]
HG = list(dict.fromkeys(HG))
Mh = M.loc[HG].dropna(axis=1, thresh=int(0.6 * len(HG)))
labels, Z, order = TH.donor_clusters(Mh, min_size=3)
print(f"{Mh.shape[0]} genes x {Mh.shape[1]} donors -> {labels.nunique()} donor clusters "
      f"{labels.value_counts().sort_index().to_dict()}")
if labels.nunique() == 1:
    print("\n*** NO DONOR GROUPING. No cut of the dendrogram into clusters of at least "
          "3 donors beat the silhouette criterion, so on these genes the 52 patients do "
          "not fall into subtypes: the heterogeneity measured above is a continuum, not "
          "a set of groups. The association tests below are then trivially one-sided, "
          "and F1 still shows the per-donor structure. A split into one outlier and "
          "everyone else, which an unconstrained silhouette would have returned, is not "
          "a subtype. ***")

ann = DON.reindex(labels.index)[["dx2", "study", "sex", "age", "burden", "tx_ecp",
                                 "tx_hdaci", "is_cultured", "n_prior_lines"]].copy()
ann["cluster"] = labels
print("\ndonor clusters against the clinical fields:")
for col in ["dx2", "study", "sex", "tx_ecp", "tx_hdaci", "is_cultured"]:
    ct = pd.crosstab(ann["cluster"], ann[col])
    if ct.shape[1] < 2:
        continue
    p = stats.chi2_contingency(ct)[1] if ct.values.min() >= 0 else np.nan
    print(f"\n  {col}  (chi2 p = {p:.3g})")
    print(ct.to_string())
for col in ["age", "burden", "n_prior_lines"]:
    g = [ann.loc[ann["cluster"].eq(k), col].dropna() for k in sorted(ann["cluster"].unique())]
    g = [x for x in g if len(x) >= 3]
    if len(g) >= 2:
        print(f"\n  {col}: medians {[round(float(x.median()), 3) for x in g]}  "
              f"Kruskal p = {stats.kruskal(*g).pvalue:.3g}")

print("\nper-cluster mean effect for the leading antigens -- what distinguishes the groups:")
prof = Mh.T.groupby(labels).mean().T
prof["spread"] = prof.max(1) - prof.min(1)
print(prof.sort_values("spread", ascending=False).head(20).round(3).to_string())

ann.to_csv(ST.tab("donor_clusters"))
prof.to_csv(ST.tab("donor_cluster_profiles"))
print("\nwrote", ST.tab("donor_clusters"), "and", ST.tab("donor_cluster_profiles"))

In [ ]:
# ============================================================================
# §4d F1  The donor x gene matrix with the clinical annotation beside it
# ============================================================================
# Three orderings of the same matrix. F1: donors in effect-profile order (as before). F1b: the
# same, with the donor dendrogram drawn, so the eye can see how deep each split is. F1c: genes
# clustered into programs that rise and fall together across patients, donors by burden.
from scipy.cluster.hierarchy import dendrogram

AN = [("cluster", ann["cluster"].astype(str)), ("dx", ann["dx2"].astype(str)),
      ("ECP", ann["tx_ecp"].map({True: "yes", False: "no"}).astype(str)),
      ("sex", ann["sex"].astype(str))]
NUM = [("burden", ann["burden"]), ("age", ann["age"])]
STARS = dict(zip(re_tab["gene"], re_tab["q_Q"]))
DENDRO = dict(no_labels=True, color_threshold=0, above_threshold_color="#888888")


def _strip(ax, name, vv, categorical):
    if categorical:
        cats = [c for c in pd.unique(vv.dropna()) if c != "nan"]
        cmap = dict(zip(cats, plt.get_cmap("tab20")(np.linspace(0, 1, max(len(cats), 2)))))
        ax.imshow([[1]], visible=False)
        for x, val in enumerate(vv):
            ax.add_patch(plt.Rectangle((x - .5, -.5), 1, 1,
                                       facecolor=cmap.get(val, "#eeeeee"), edgecolor="none"))
        # key, each word in its own colour, so the strip can be read without a lookup
        _x = 1.005
        for c in cats:
            ax.text(_x, 0, str(c), transform=ax.transAxes, fontsize=4.5, va="center",
                    color=cmap.get(c, "#888888"))
            _x += 0.012 + 0.009 * len(str(c))
    else:
        z = (vv - vv.min()) / (vv.max() - vv.min() or 1)
        for x, val in enumerate(z):
            ax.add_patch(plt.Rectangle((x - .5, -.5), 1, 1,
                                       facecolor=plt.get_cmap("Greys")(0.15 + 0.8 * val)
                                       if np.isfinite(val) else "#ffffff", edgecolor="none"))
        ax.text(1.005, 0, f"{vv.min():.3g}-{vv.max():.3g}", transform=ax.transAxes,
                fontsize=4.5, va="center", color="#555555")
    ax.set_xlim(-.5, len(vv) - .5); ax.set_ylim(-.5, .5)
    ax.set_yticks([0]); ax.set_yticklabels([name], fontsize=5.5)
    ax.set_xticks([]); [s.set_visible(False) for s in ax.spines.values()]


def draw_heat(Md, title, xlabel, col_Z=None, row_Z=None, row_groups=None):
    """Annotation strips over a gene x donor heatmap, with optional dendrograms. `col_Z` /
    `row_Z` must be linkages whose leaf order is already the column / row order of `Md`."""
    ordc = list(Md.columns)
    W = 0.13 * Md.shape[1] + 3.4 + (1.0 if row_Z is not None else 0)
    H = 0.17 * len(Md) + 2.6 + (0.5 if col_Z is not None else 0)
    ratios = ([2.0] if col_Z is not None else []) + [0.5] * (len(AN) + len(NUM)) \
        + [len(Md) * 0.5]
    fig = plt.figure(figsize=(W, H))
    gs = fig.add_gridspec(len(ratios), 1, height_ratios=ratios, hspace=0.12, top=0.955,
                          bottom=0.10, left=(1.4 + (1.0 if row_Z is not None else 0)) / W,
                          right=0.88)
    k = 0
    if col_Z is not None:
        axd = fig.add_subplot(gs[0]); k = 1
        dendrogram(col_Z, ax=axd, **DENDRO)
        axd.set_xlim(0, 10 * len(ordc)); axd.set_axis_off()
    for i, (name, v) in enumerate(AN + NUM):
        _strip(fig.add_subplot(gs[k + i]), name, v.reindex(ordc),
               categorical=name in [a for a, _ in AN])

    ax = fig.add_subplot(gs[-1])
    TF.donor_gene_heat(ax, Md, stars_by=STARS, cbar_label="log2FC malignant / reactive")
    ax.set_xlabel(xlabel)
    if row_Z is not None:
        fig.canvas.draw()
        p = ax.get_position()
        lab = 0.75 / W                                  # room for the gene names
        if row_groups is not None:
            axs = fig.add_axes([p.x0 - lab - 0.12 / W, p.y0, 0.10 / W, p.height])
            g = row_groups.reindex(Md.index)
            cols = plt.get_cmap("tab10").colors
            for y, gi in enumerate(g):
                axs.add_patch(plt.Rectangle((0, y - .5), 1, 1, edgecolor="none",
                                            facecolor=cols[(int(gi) - 1) % 10]))
            axs.set_xlim(0, 1); axs.set_ylim(len(Md) - .5, -.5); axs.set_axis_off()
        axr = fig.add_axes([p.x0 - lab - 0.95 / W, p.y0, 0.80 / W, p.height])
        dendrogram(row_Z, ax=axr, orientation="right", **DENDRO)
        axr.set_ylim(10 * len(Md), 0); axr.set_axis_off()
    fig.suptitle(title, x=0.01, y=0.995, ha="left", fontsize=9)
    return fig


CAP = "Shortlist + 40 most heterogeneous genes. Stars: BH heterogeneity Q."

# F1 -- donors in effect-profile order
ordc = [d for d in order if d in Mh.columns]
Md = Mh[ordc]
fig = draw_heat(Md, "Per-donor log2FC, donors clustered", "donor, effect-profile order")
TF.save(fig, "F1_donor_clustermap", caption_text=CAP)
TF.save_source(Md.reset_index(), "F1_donor_clustermap_source")
TF.logfig("F1_donor_clustermap", "donor x gene effects with clinical annotation", Md.size)

# F1b -- same order, donor dendrogram shown
fig = draw_heat(Md, "Per-donor log2FC, by sample similarity",
                "donor (average linkage, correlation distance)", col_Z=Z)
TF.save(fig, "F1b_donor_dendrogram", caption_text=CAP)
TF.logfig("F1b_donor_dendrogram", "donor x gene effects, donor dendrogram", Md.size)

# F1c -- genes clustered into co-varying programs, donors by burden
glab, gZ, gorder = TH.donor_clusters(Mh.T, min_size=3)
glab = glab.rename("program")
_b = ann["burden"].reindex(Mh.columns).sort_values()
Mg = Mh.loc[gorder, list(_b.index)]
_np = glab.nunique()
fig = draw_heat(Mg, f"Per-donor log2FC, {_np} gene programs" if _np > 1
                else "Per-donor log2FC, genes clustered (no distinct programs)",
                f"donor, ordered by malignant share of CD4 ({_b.min():.0%} → {_b.max():.0%})",
                row_Z=gZ, row_groups=glab if _np > 1 else None)
TF.save(fig, "F1c_gene_programs", caption_text=CAP + (" Left strip: gene program." if _np > 1
                                                      else " No silhouette cut with ≥3 genes "
                                                           "per program."))
TF.save_source(Mg.reset_index(), "F1c_gene_programs_source")
TF.logfig("F1c_gene_programs", "donor x gene effects, genes clustered into programs", Mg.size)

if glab.nunique() == 1:
    print("no gene grouping beat the silhouette criterion (min 3 genes per program): the "
          "dendrogram is shown, but the genes do not split into programs")
prog = glab.reindex(gorder).reset_index().rename(columns={"index": "gene"})
prog.columns = ["gene", "program"]
print("gene programs:")
for k_, g_ in prog.groupby("program")["gene"]:
    print(f"  {k_}: {', '.join(g_)}")
_pm = Mh.groupby(glab).mean()
print("\nper-program effect across donors:")
print(pd.DataFrame({"n_genes": glab.value_counts().sort_index(),
                    "median_lfc": _pm.median(1), "frac_donors_up_0.5": (_pm > 0.5).mean(1),
                    "rho_burden": _pm.T.corrwith(ann["burden"].reindex(_pm.columns),
                                                 method="spearman")})
      .to_string(float_format=lambda v: f"{v:.3g}"))
prog.to_csv(ST.tab("gene_programs"), index=False)
print("wrote", ST.tab("gene_programs"))

In [ ]:
# ============================================================================
# §4e F2  Eligibility: how many patients, at what tumour-positivity cut
# ============================================================================
THRESH = (0.05, 0.10, 0.25, 0.50, 0.75)
elig = TH.eligibility(long, MALIG, thresholds=THRESH, donors=PAIRED)
elig = elig.merge(re_tab[["gene", "mu", "I2"]], on="gene", how="left")
elig["on_shortlist"] = elig["gene"].isin(SHORT)
_e = ["gene", "n_donors", "median_pct_expr"] + \
     [f"frac_donors_ge{int(t * 100)}" for t in THRESH] + ["mu", "I2"]

print("43's shortlist, as a recruitment statement:")
print(elig[elig["on_shortlist"]].sort_values("frac_donors_ge25", ascending=False)[_e]
      .to_string(index=False, float_format=lambda v: f"{v:.3g}"))

print("\nbest coverage at a 25% tumour-cell cut, over every screened gene:")
print(elig.sort_values("frac_donors_ge25", ascending=False).head(25)[_e]
      .to_string(index=False, float_format=lambda v: f"{v:.3g}"))

TOPE = elig[elig["on_shortlist"]].sort_values("frac_donors_ge25",
                                              ascending=False)["gene"].head(10).tolist()
fig, ax = plt.subplots(figsize=(5.0, 3.2))
cmap = plt.get_cmap("viridis")
for i, gene in enumerate(TOPE):
    r_ = elig[elig["gene"].eq(gene)].iloc[0]
    y = [r_[f"frac_donors_ge{int(t * 100)}"] for t in THRESH]
    ax.step(THRESH, y, where="post", lw=1.2, color=cmap(i / max(len(TOPE) - 1, 1)),
            label=gene)
    ax.scatter(THRESH, y, s=8, color=cmap(i / max(len(TOPE) - 1, 1)), zorder=3)
ax.set_xlabel("required fraction of a patient's malignant CD4 cells positive")
ax.set_ylabel("fraction of patients eligible")
ax.set_ylim(0, 1.02)
ax.legend(fontsize=5.5, frameon=False, ncol=2)
ax.set_title("Patients eligible by tumour-positivity cut", loc="left", fontsize=7.5)
TF._clean(ax)
fig.tight_layout()
TF.save(fig, "F2_eligibility", caption_text=(
    f"Transcript detection, 10k-HVG object. {len(PAIRED)} donors."))
TF.save_source(elig, "F2_eligibility_source")
TF.logfig("F2_eligibility", "eligibility curves for the shortlist", len(elig))

In [ ]:
# ============================================================================
# §4f F3  Consistency against size: which antigens are cohort-wide and which are subgroup
# ============================================================================
d = re_tab.merge(cp[["gene", "frac_outlier"]], on="gene", how="left")
d = d[d["n_donors"] >= MIN_DONORS]
fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.axvline(0, lw=.5, c="k")
ax.plot([0, d["mu"].max()], [0, d["mu"].max()], lw=.5, c="#bbbbbb", ls="--")
ax.scatter(d["mu"], d["tau"], s=6, c="#d5d8dc", linewidths=0)
k = d["on_shortlist"]
sc = ax.scatter(d.loc[k, "mu"], d.loc[k, "tau"], s=24, c=d.loc[k, "frac_outlier"],
                cmap="magma_r", vmin=0, vmax=0.4, linewidths=0)
ax.margins(x=0.12)
_txt = [ax.text(r_["mu"], r_["tau"], r_["gene"], fontsize=5.5)
        for _, r_ in d[k].sort_values("mu", ascending=False).head(14).iterrows()]
try:
    from adjustText import adjust_text
    adjust_text(_txt, ax=ax, expand=(1.15, 1.3),
                arrowprops=dict(arrowstyle="-", lw=0.3, color="#999999"))
except ImportError:
    pass
cb = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
cb.set_label("fraction of donors in the outlier tail", fontsize=6)
cb.ax.tick_params(labelsize=5.5)
ax.set_xlabel("pooled effect (random-effects mu), log2FC")
ax.set_ylabel(r"$\tau$ — between-patient SD of the effect (log2FC)")
ax.set_title("Pooled effect vs between-patient spread", loc="left", fontsize=7.5)
TF._clean(ax)
fig.tight_layout()
TF.save(fig, "F3_effect_vs_heterogeneity", caption_text=(
    "Grey: all genes; coloured: shortlist. Dashed: τ = μ."))
TF.save_source(d, "F3_effect_vs_heterogeneity_source")
TF.logfig("F3_effect_vs_heterogeneity", "pooled effect vs I2", len(d))

print("the quadrant that matters for a single-agent therapy -- large and consistent:")
q1 = d[(d["mu"] > 0.5) & (d["tau"] < d["tau"].median()) & (d["pi_lo"] > 0)]
print(q1.sort_values("mu", ascending=False)
      [["gene", "mu", "tau", "pi_lo", "pi_hi", "frac_donors_up", "on_shortlist"]]
      .head(25).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

print("\nand the quadrant that argues for patient selection -- large but not in everyone:")
q2 = d[(d["mu"] > 0.5) & (d["tau"] >= d["tau"].median())]
print(q2.sort_values("mu", ascending=False)
      [["gene", "mu", "tau", "pi_lo", "pi_hi", "frac_donors_up", "frac_outlier",
        "on_shortlist"]].head(25)
      .to_string(index=False, float_format=lambda v: f"{v:.3g}"))

In [ ]:
# ============================================================================
# §5  Dossier
# ============================================================================
def SEC(t): print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)
def kv(k, v): print(f"  {k:36s} {v}")

SEC("1 - COHORT")
kv("paired donors", len(DON))
kv("people", DON["person"].nunique())
print()
print(DON["dx"].value_counts().to_string())
kv("\n  studies", DON["study"].nunique())
kv("donors with every model covariate", len(X))

SEC("2 - WHAT WAS TESTED (TIER 1)")
for t, n in (("MF vs SS", f"{int(dx_map.eq('MF').sum())} vs {int(dx_map.eq('SS').sum())}"),
             ("ECP vs not", f"{int((ecp_map == 'ECP').sum())} vs "
                            f"{int((ecp_map == 'no ECP').sum())}"),
             ("sex", f"{int(DON['sex'].eq('F').sum())} F / "
                     f"{int(DON['sex'].eq('M').sum())} M"),
             ("age", f"{int(DON['age'].notna().sum())} donors, median "
                     f"{DON['age'].median():.0f}"),
             ("burden", f"{int(DON['burden'].notna().sum())} donors, "
                        f"{DON['burden'].min():.0%}-{DON['burden'].max():.0%}")):
    kv(t, n)
print()
print(pd.Series({t: int((reg[f"q_{t}"] < 0.05).sum()) for t in X.columns},
                name="genes at q<0.05").to_string())

SEC("3 - HETEROGENEITY")
kv("genes pooled", len(re_tab))
kv("median I2 (saturated)", f"{re_tab['I2'].median():.2f}")
kv("median tau", f"{re_tab['tau'].median():.2f}")
kv("pooled effect > 0.3", int((re_tab["mu"] > 0.3).sum()))
kv("  of those, PI excludes zero", int(((re_tab["mu"] > 0.3) & (re_tab["pi_lo"] > 0)).sum()))
kv("donor clusters", f"{labels.nunique()} {labels.value_counts().sort_index().to_dict()}")

SEC("4 - WHAT THIS IS NOT")
for line in [
    "A treatment-effect study. Treatment is confounded with study; ECP is the only agent with "
    "two arms and even that is mostly buus and ren.",
    "A disease comparison with power: 8 MF donors against 37 SS, MF spread over 4 studies.",
    "A stage analysis: 2 early-stage donors.",
    "A response or survival analysis: those fields are empty in this cohort.",
    "A protein result. Every number is transcript, in a 10,000-gene HVG subset.",
]:
    print("  - " + line)

SEC("5 - OUTPUTS")
TF.write_index("figure_index")
for p in sorted(ST.TAB_DIR.glob(f"{PREFIX}*")) + sorted(ST.FIG_DIR.glob(f"{PREFIX}*")):
    print("  ", p.relative_to(NB_DIR))

In [ ]:
# ============================================================================
# §6  The complete numerical result, as one block to hand to another model
# ============================================================================
_ctx = f"""
Compartment: MF/CTCL **blood**. This notebook stratifies the malignant-vs-reactive CD4 antigen
axis that notebook 43 established, and measures how much it varies between patients. It derives
no new screen: it reads 43's cleaned cohort and tables.

Cohort: {len(DON)} donors ({DON['person'].nunique()} people, {DON['study'].nunique()} studies)
carrying BOTH a malignant and a reactive CD4 arm of >= 20 cells -- 37 Sezary, 8 MF, 5 unresolved,
2 MF/SS. The geskin2026 ex vivo drug wells and duplicate persons were removed in 43 section 0b,
and five healthy donors the atlas labelled SS were moved out of the reactive arm.

Per-donor effect: for each gene and donor, log2((mean malignant + 0.05)/(mean reactive + 0.05))
on log1p CP10K. Everything below is a function of that donor x gene matrix.

Stratified tests: the paired Wilcoxon is rerun inside each stratum, and `p_interaction` compares
the per-donor effects between strata (BH as `q_interaction`). Per-gene OLS regresses the donor
effect on diagnosis, ECP, sex, age and burden at once.

Heterogeneity: DerSimonian-Laird random effects (`mu` pooled effect, `tau` between-patient SD,
`I2`, `pi_lo`/`pi_hi` prediction interval, `q_Q`); COPA and the outlier sum with a
{N_PERM}-draw sign-flip null; two-component Gaussian mixture BIC; hierarchical donor clustering
with a silhouette cut and at least 3 donors per cluster.

Power limits that apply to every result: treatment is almost entirely confounded with study (ECP
is the only agent with two usable arms, and it is concentrated in buus and ren); there are 2
early-stage donors against 35 advanced; response and survival are essentially absent. I2
saturates near 1 because each donor contributes thousands of cells, so rank consistency on `tau`,
not on I2. Everything is transcript, in a 10,000-gene HVG subset.
"""
_sec = [
    {"name": "§1 Power table",
     "what": "coverage of every candidate stratification over the paired donors.",
     "how": "count of non-missing values per field, split by diagnosis.",
     "table": power, "file": "tables/tgt44_power_table.csv"},
    {"name": "§1 Treatment at sampling",
     "what": "donors on each treatment class, and in which studies.",
     "how": "membership of the curated clin_current_tx_classes string; a donor can be on "
            "several.", "table": tx, "file": "tables/tgt44_treatment_counts.csv"},
    {"name": "§2a Disease, MF vs SS, on 43's shortlist",
     "what": "each leading antigen's effect inside MF and inside SS, and whether they differ.",
     "how": "paired Wilcoxon within stratum; delta_lfc and q_interaction compare the per-donor "
            "effects. 8 MF donors: a consistency check, not a finding.",
     "table": dx_tab[dx_tab["on_shortlist"]].sort_values("log2fc_all", ascending=False),
     "file": "tables/tgt44_strat_disease.csv", "max_rows": 40},
    {"name": "§2b ECP, on 43's shortlist",
     "what": "the same split by extracorporeal photopheresis at sampling.",
     "how": "as above; the study-blocked (van Elteren) version is the one to quote and is in "
            "the file named below.",
     "table": ew[ew["on_shortlist"]], "file": "tables/tgt44_strat_ecp_blocked.csv",
     "max_rows": 40},
    {"name": "§2c Donor-level regression",
     "what": "which donor properties move each gene's effect, all terms in one model.",
     "how": "OLS per gene over donors, terms " + ", ".join(X.columns) + "; BH per term. Study "
            "is deliberately out of the design and is reported as its own ANOVA.",
     "table": pd.Series({t: int((reg[f"q_{t}"] < 0.05).sum()) for t in X.columns},
                        name="genes_at_q_lt_0.05"),
     "file": "tables/tgt44_strat_regression.csv"},
    {"name": "§3 Tier 2 descriptive splits",
     "what": "effect sizes on each side of the splits too small to test.",
     "how": "median per-donor effect per side with a 400-draw bootstrap CI. No p-value.",
     "table": t2, "file": "tables/tgt44_tier2_descriptive.csv", "max_rows": 60},
    {"name": "§4a Random-effects meta-analysis, 43's shortlist",
     "what": "pooled effect and between-patient spread for each leading antigen.",
     "how": "DerSimonian-Laird; `pi_lo > 0` is what 'works in every patient' means.",
     "table": re_tab[re_tab["on_shortlist"]].sort_values("tau"),
     "file": "tables/tgt44_heterogeneity_meta.csv", "max_rows": 40},
    {"name": "§4a Most consistent antigens cohort-wide",
     "what": "pooled effect above 0.3 with a prediction interval excluding zero.",
     "how": "same model, ranked by mu.",
     "table": con[["gene", "n_donors", "mu", "tau", "pi_lo", "pi_hi", "frac_donors_up",
                   "on_shortlist"]], "max_rows": 40},
    {"name": "§4b COPA, subgroup antigens",
     "what": "genes extreme in a minority of patients, which a mean test averages away.",
     "how": "median/MAD-standardised 75th percentile, sign-flip null, BH.",
     "table": cp.sort_values("copa_p75", ascending=False).head(40)[
         [c for c in ("gene", "copa_p75", "frac_outlier", "q_copa", "mu", "log2fc_paired", "q",
                      "on_shortlist") if c in cp.columns]],
     "file": "tables/tgt44_heterogeneity_copa.csv"},
    {"name": "§4b Bimodality of the per-donor effect",
     "what": "whether a gene's donor effects fall into two groups rather than one spread.",
     "how": "one- versus two-component Gaussian mixture BIC; delta_bic > 0 favours two "
            "components. A weak instrument at this donor count -- read it beside I2 and COPA.",
     "table": bm.head(25), "file": 'tables/tgt44_heterogeneity_bimodality.csv'},
    {"name": "§4c Donor clusters against the clinical fields",
     "what": "whether the patients fall into groups on these antigens.",
     "how": "hierarchical clustering of the donor x gene matrix, silhouette cut, min 3 donors.",
     "table": ann, "file": "tables/tgt44_donor_clusters.csv", "max_rows": 80},
    {"name": "§4c Per-cluster mean effect for the leading antigens",
     "what": "what separates the donor clusters, where there are any.",
     "how": "mean per-donor effect inside each cluster; `spread` is max minus min across "
            "clusters.",
     "table": prof.sort_values("spread", ascending=False).head(25), "file": 'tables/tgt44_donor_cluster_profiles.csv'},
    {"name": "§4e Eligibility",
     "what": "the fraction of patients in whom at least X% of malignant CD4 cells are positive.",
     "how": "transcript detection per donor, thresholded at 5/10/25/50/75%.",
     "table": elig[elig["on_shortlist"]], "max_rows": 40},
    {"name": "§4f Large and consistent vs large and selective",
     "what": "the two quadrants a development decision turns on.",
     "how": "mu > 0.5 split on tau against its cohort median and on whether the prediction "
            "interval excludes zero.",
     "table": pd.concat([q1.assign(quadrant="large_and_consistent"),
                         q2.assign(quadrant="large_but_variable")])
                [["gene", "quadrant", "mu", "tau", "pi_lo", "pi_hi", "frac_donors_up",
                  "on_shortlist"]], "max_rows": 50},
]
TF.llm_export("44", "Blood stratification and heterogeneity", _ctx, _sec)